# Stage 0 — Data Preparation

Cleans and deduplicates raw text data before topic modelling.

**Runtime:** Google Colab + Google Drive  
**Storage:** All inputs and outputs are read from / written to a project folder on Google Drive.  
**Library:** [`multilingual-topic-modeling`](https://github.com/ay94/multilingual-topic-modeling)

**Outputs:**
- `cleaned_dataset.jsonl.gz` — cleaned, deduplicated messages
- `cleaned_dataset_duplicates.jsonl.gz` — duplicate records (for later propagation)

See [`docs/workflow/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/workflow) for the full methodology.

## Setup

In [ ]:
# Mount Google Drive — all data lives here
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install multilingual-topic-modeling --quiet

In [ ]:
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from multilingual_topic import FileHandler, TextPreprocessor

## Configuration

Set your project folder path on Google Drive and the column names for your dataset.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/YOUR_PROJECT/Topic Modelling Workflow'
TEXT_COL = 'text'        # column containing the raw message text
ID_COL   = 'id'          # unique message identifier column
LANG_COL = None          # set to a column name to filter by language, or None to skip
LANG_KEEP = 'en'         # language value to keep (used only if LANG_COL is set)
# ──────────────────────────────────────────────────────────────────────────────

fh = FileHandler(DRIVE_FOLDER)

## 1. Load data

In [ ]:
# Load your raw dataset — adjust path and format as needed
dataset = pd.read_csv(fh.create_filename('data/raw_data.csv'))
print(f'Loaded: {len(dataset):,} rows')
dataset.head()

## 2. Filter by language (optional)

In [ ]:
if LANG_COL:
    dataset = dataset[dataset[LANG_COL] == LANG_KEEP]
    print(f'After language filter ({LANG_KEEP}): {len(dataset):,} rows')

## 3. Clean text

In [ ]:
pp = TextPreprocessor()
dataset['cleaned_text'] = dataset[TEXT_COL].apply(pp.preprocess)
dataset = dataset[dataset['cleaned_text'].str.strip().str.len() > 0]
print(f'After cleaning: {len(dataset):,} rows')

## 4. Deduplicate

Duplicates are separated rather than dropped — they will be labelled later by propagating topic assignments from their unique counterparts.

> **Note on deduplication timing:** if coordination detection is an analytical objective, deduplication should happen *after* drawing an analytical sample — not before. Repeated messages reflect coordinated behaviour, and removing them before sampling eliminates that signal. See [`docs/workflow/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/workflow) for detail.

In [ ]:
# Count occurrences of each cleaned message
counts = dataset.groupby('cleaned_text').size().reset_index(name='count')
dataset = dataset.merge(counts, on='cleaned_text')

# Split: unique messages vs duplicates
unique_dataset = dataset.drop_duplicates(subset='cleaned_text')
duplicates_dataset = dataset[dataset['count'] > 1]

print(f'Unique messages : {len(unique_dataset):,}')
print(f'Duplicate records: {len(duplicates_dataset):,}')
print(f'Duplication rate : {(1 - len(unique_dataset)/len(dataset))*100:.1f}%')

## 5. Save outputs

In [ ]:
unique_dataset.to_json(
    fh.create_filename('data/cleaned_dataset.jsonl.gz'),
    orient='records', lines=True,
)

duplicates_dataset.to_json(
    fh.create_filename('data/cleaned_dataset_duplicates.jsonl.gz'),
    orient='records', lines=True,
)

print('Saved.')
print(f"  → {fh.create_filename('data/cleaned_dataset.jsonl.gz')}")
print(f"  → {fh.create_filename('data/cleaned_dataset_duplicates.jsonl.gz')}")